In [1]:
from pathlib import Path
import pandas as pd

## Configuration


In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "src" / "data" / "raw" / "SPY" / "2010" / "spy_eod_201001.txt" ### name of file

## Load

In [3]:
df = pd.read_csv(DATA_PATH, sep=",")
df.columns = df.columns.str.strip()

print("=" * 80)
print("RAW DATA PROFILE")
print("=" * 80)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

RAW DATA PROFILE
Rows: 18,668
Columns: 33


C:\Users\Admin\AppData\Local\Temp\ipykernel_12072\1456566105.py:1: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, sep=",")


## Columns

In [4]:
print("=" * 80)
print("COLUMN NAMES")
print("=" * 80)

for i, col in enumerate(df.columns):
    print(f"{i:02d}: {repr(col)}")

COLUMN NAMES
00: '[QUOTE_UNIXTIME]'
01: '[QUOTE_READTIME]'
02: '[QUOTE_DATE]'
03: '[QUOTE_TIME_HOURS]'
04: '[UNDERLYING_LAST]'
05: '[EXPIRE_DATE]'
06: '[EXPIRE_UNIX]'
07: '[DTE]'
08: '[C_DELTA]'
09: '[C_GAMMA]'
10: '[C_VEGA]'
11: '[C_THETA]'
12: '[C_RHO]'
13: '[C_IV]'
14: '[C_VOLUME]'
15: '[C_LAST]'
16: '[C_SIZE]'
17: '[C_BID]'
18: '[C_ASK]'
19: '[STRIKE]'
20: '[P_BID]'
21: '[P_ASK]'
22: '[P_SIZE]'
23: '[P_LAST]'
24: '[P_DELTA]'
25: '[P_GAMMA]'
26: '[P_VEGA]'
27: '[P_THETA]'
28: '[P_RHO]'
29: '[P_IV]'
30: '[P_VOLUME]'
31: '[STRIKE_DISTANCE]'
32: '[STRIKE_DISTANCE_PCT]'


## Datatypes

In [5]:
print("=" * 80)
print("DATA TYPES")
print("=" * 80)

print(df.dtypes)

DATA TYPES
[QUOTE_UNIXTIME]           int64
[QUOTE_READTIME]          object
[QUOTE_DATE]              object
[QUOTE_TIME_HOURS]       float64
[UNDERLYING_LAST]        float64
[EXPIRE_DATE]             object
[EXPIRE_UNIX]              int64
[DTE]                    float64
[C_DELTA]                float64
[C_GAMMA]                float64
[C_VEGA]                 float64
[C_THETA]                float64
[C_RHO]                  float64
[C_IV]                    object
[C_VOLUME]                object
[C_LAST]                  object
[C_SIZE]                  object
[C_BID]                   object
[C_ASK]                   object
[STRIKE]                 float64
[P_BID]                   object
[P_ASK]                   object
[P_SIZE]                  object
[P_LAST]                  object
[P_DELTA]                float64
[P_GAMMA]                float64
[P_VEGA]                 float64
[P_THETA]                float64
[P_RHO]                  float64
[P_IV]                    object

## String-based Missing Values

In [6]:
print("=" * 80)
print("BLANK / STRING MISSING VALUES")
print("=" * 80)

for col in df.columns:
    if df[col].dtype == "object":
        blank_count = (
            df[col].astype("string").str.strip()
            .eq("")
            .sum()
        )

        if blank_count > 0:
            pct = blank_count / len(df) * 100

            print(
                f"{col:<25} "
                f"{blank_count:>8,} "
                f"({pct:6.2f}%)"
            )

BLANK / STRING MISSING VALUES
[C_IV]                       1,707 (  9.14%)
[C_VOLUME]                   4,522 ( 24.22%)
[C_LAST]                     2,979 ( 15.96%)
[C_BID]                      2,979 ( 15.96%)
[C_ASK]                      2,979 ( 15.96%)
[P_BID]                      2,979 ( 15.96%)
[P_ASK]                      2,979 ( 15.96%)
[P_LAST]                     2,979 ( 15.96%)
[P_IV]                         636 (  3.41%)
[P_VOLUME]                   4,755 ( 25.47%)


## True (Pandas-level) Missing Values

In [7]:
print("=" * 80)
print("PANDAS NaN / NA")
print("=" * 80)

missing = df.isna().sum()
missing = missing[missing > 0]

if missing.empty:
    print("No pandas-level missing values detected.")
else:
    print(missing)

PANDAS NaN / NA
No pandas-level missing values detected.


## Date Coverage

In [8]:
print("=" * 80)
print("DATE COVERAGE")
print("=" * 80)

quote_dates = pd.to_datetime(
    df["[QUOTE_DATE]"], errors="coerce"
)

expire_dates = pd.to_datetime(
    df["[EXPIRE_DATE]"], errors="coerce"
)

print(f"Quote start:  {quote_dates.min()}")
print(f"Quote end:    {quote_dates.max()}")
print(f"Expiry start: {expire_dates.min()}")
print(f"Expiry end:   {expire_dates.max()}")

DATE COVERAGE
Quote start:  2010-01-04 00:00:00
Quote end:    2010-01-29 00:00:00
Expiry start: 2010-01-15 00:00:00
Expiry end:   2012-12-21 00:00:00


## Quote Timestamp

In [9]:
print("=" * 80)
print("QUOTE TIMESTAMPS")
print("=" * 80)

print(
    "Unique QUOTE_UNIXTIME:", df["[QUOTE_UNIXTIME]"].nunique()
)

print(
    "Unique QUOTE_READTIME:", df["[QUOTE_READTIME]"].nunique()
)

print(
    "Unique QUOTE_TIME_HOURS:", df["[QUOTE_TIME_HOURS]"].nunique()
)

print(
    "\nQUOTE_TIME_HOURS distribution:"
)

print(
    df["[QUOTE_TIME_HOURS]"].value_counts(dropna=False).sort_index()
)

QUOTE TIMESTAMPS
Unique QUOTE_UNIXTIME: 19
Unique QUOTE_READTIME: 19
Unique QUOTE_TIME_HOURS: 1

QUOTE_TIME_HOURS distribution:
[QUOTE_TIME_HOURS]
16.0    18668
Name: count, dtype: int64


## Underlying Price Consistency

In [10]:
print("=" * 80)
print("UNDERLYING PRICE CONSISTENCY")
print("=" * 80)

underlying_per_timestamp = (
    df.groupby("[QUOTE_UNIXTIME]")["[UNDERLYING_LAST]"]
    .nunique()
)

print(
    "Timestamps with >1 underlying price:",
    (underlying_per_timestamp > 1).sum()
)

print(
    "Maximum distinct underlying prices per timestamp:",
    underlying_per_timestamp.max()
)

UNDERLYING PRICE CONSISTENCY
Timestamps with >1 underlying price: 0
Maximum distinct underlying prices per timestamp: 1


## Contract Uniqueness

In [11]:
print("=" * 80)
print("CONTRACT UNIQUENESS")
print("=" * 80)

contract_keys = [
    "[QUOTE_UNIXTIME]",
    "[EXPIRE_UNIX]",
    "[STRIKE]",
]

duplicate_contract_rows = df.duplicated(
    subset=contract_keys, keep=False
)

print(
    "Rows participating in duplicate contract keys:",
    duplicate_contract_rows.sum()
)

print(
    "Unique contract keys:",
    df[contract_keys].drop_duplicates().shape[0]
)

print(
    "Total rows:",
    len(df)
)

CONTRACT UNIQUENESS
Rows participating in duplicate contract keys: 0
Unique contract keys: 18668
Total rows: 18668


## DTE

In [12]:
print("=" * 80)
print("DTE")
print("=" * 80)

print(df["[DTE]"].describe())

print(
    "\nDTE <= 0:",
    (df["[DTE]"] <= 0).sum()
)

print(
    "DTE < 7:",
    (df["[DTE]"] < 7).sum()
)

print(
    "DTE < 30:",
    (df["[DTE]"] < 30).sum()
)

print(
    "DTE > 365:",
    (df["[DTE]"] > 365).sum()
)

DTE
count    18668.000000
mean       215.235577
std        210.975353
min          0.000000
25%         63.960000
50%        159.960000
75%        326.000000
max       1082.000000
Name: [DTE], dtype: float64

DTE <= 0: 110
DTE < 7: 550
DTE < 30: 1870
DTE > 365: 1447


## Underlying & Strike Prices

In [13]:
print("=" * 80)
print("UNDERLYING / STRIKE")
print("=" * 80)

print("Underlying:")
print(df["[UNDERLYING_LAST]"].describe())

print("\nStrike:")
print(df["[STRIKE]"].describe())

UNDERLYING / STRIKE
Underlying:
count    18668.000000
mean       112.390695
std          2.460518
min        107.430000
25%        109.780000
50%        113.660000
75%        114.540000
max        115.060000
Name: [UNDERLYING_LAST], dtype: float64

Strike:
count    18668.000000
mean       103.704253
std         32.281923
min         20.000000
25%         79.000000
50%        103.000000
75%        127.000000
max        210.000000
Name: [STRIKE], dtype: float64


## Raw Object Cardinality

In [14]:
print("=" * 80)
print("OBJECT COLUMN CARDINALITY")
print("=" * 80)

object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    print(
        f"{col:<25} "
        f"{df[col].nunique(dropna=False):>8,} unique values"
    )

OBJECT COLUMN CARDINALITY
[QUOTE_READTIME]                19 unique values
[QUOTE_DATE]                    19 unique values
[EXPIRE_DATE]                   14 unique values
[C_IV]                      13,444 unique values
[C_VOLUME]                     548 unique values
[C_LAST]                     1,591 unique values
[C_SIZE]                     9,784 unique values
[C_BID]                      3,373 unique values
[C_ASK]                      3,430 unique values
[P_BID]                      2,790 unique values
[P_ASK]                      2,779 unique values
[P_SIZE]                    11,913 unique values
[P_LAST]                     1,370 unique values
[P_IV]                      14,590 unique values
[P_VOLUME]                     653 unique values


## Sample Raw Values from Important Columns

In [15]:
print("=" * 80)
print("IMPORTANT RAW VALUES")
print("=" * 80)

inspect_columns = [
    "[C_IV]",
    "[P_IV]",
    "[C_VOLUME]",
    "[P_VOLUME]",
    "[C_LAST]",
    "[P_LAST]",
    "[C_BID]",
    "[C_ASK]",
    "[P_BID]",
    "[P_ASK]",
    "[C_SIZE]",
    "[P_SIZE]",
]

for col in inspect_columns:
    print(f"{col}")
    print(df[col].value_counts(dropna=False).head(10))

IMPORTANT RAW VALUES
[C_IV]


[C_IV]
             1707
0.155800        7
0.168900        6
0.157240        6
0.154930        6
0.223960        5
0.164680        5
-0.000130       5
-0.000180       5
0.158300        5
Name: count, dtype: int64
[P_IV]
[P_IV]
             636
0.000470      18
-0.000050     18
0.000200      15
0.000100      14
0.000130      14
0.000070      14
0.000490      13
0.000400      13
-0.000070     13
Name: count, dtype: int64
[C_VOLUME]
[C_VOLUME]
0.000000     9966
             4522
1.000000      412
2.000000      353
10.000000     285
5.000000      190
20.000000     164
15.000000     157
3.000000      134
4.000000       98
Name: count, dtype: int64
[P_VOLUME]
[P_VOLUME]
0.000000      8288
              4755
1.000000       480
10.000000      422
2.000000       297
5.000000       185
20.000000      177
3.000000       163
100.000000     143
15.000000      142
Name: count, dtype: int64
[C_LAST]
[C_LAST]
0.000000    3267
            2979
0.020000     425
0.040000     189
0.010000     183
0.030000